# MeepHash-W **v2 (frozen)** - EXACT CUDA T4 benchmark (corrected)

Correctness first: the *exact* frozen-v2 hash (full BLAKE3, framed `meep_xof`, v2 dataset, S3 scratch init, complete VM, read-chain, mix-back, final walk, finalization). Not a simplified model.

**Corrections in this revision**
* batch sweep continues **past 1536** until allocation fails or throughput plateaus
* **threads-per-block sweep** (32/64/128/256)
* **long steady-state** timing: warmup + >=30 s continuous + >=5 trials, p50 and range
* **active power only** - energy-counter delta preferred; otherwise dense sampling gated to the hashing window with ramp samples discarded (the earlier 32.5 W / 119.1 H/s-per-W figure averaged ~1 s of kernel with ~9 s of setup/idle **and** used batch 1024 instead of the best batch - it is void)
* real **occupancy** via `cudaOccupancyMaxActiveBlocksPerMultiprocessor` (+ regs, local mem, clocks, temp)
* Colab CPU cell converted to `%%bash` (previously a Python SyntaxError)

**CPU denominator (measured locally, corrected):** sustained full-device 16-thread throughput is **387.9 H/s median** (range 374.5-396.2, n=5 x 30 s). The previously quoted 512.98 H/s was a 0.25 s burst and is superseded. Browser: Chrome 57.8 H/s, Firefox 51.6 H/s (single worker).


In [ ]:
!nvidia-smi
!nvcc --version | tail -2


In [ ]:
%%writefile v2_cuda_core.cuh
// EXACT MeepHash-W v2 (frozen) core — self-contained, compiles as CUDA device code AND as plain
// host C++ (so byte-exactness can be verified on the CPU before any GPU run).
//
// This is NOT a simplified performance model: it implements the full frozen v2 hash — BLAKE3
// (derive-key + XOF), the framed meep_xof, the v2 dataset word function, S3 scratchpad
// initialization, the complete VM opcode set, the mandatory dataset read-chain, mix-back, the final
// walk and finalization. It must reproduce meepow/vectors/vectors_v2.txt byte-for-byte.
//
// Build as host C++: #define MEEP_HOST_ONLY before including.
#ifndef MEEP_V2_CUDA_CORE_CUH
#define MEEP_V2_CUDA_CORE_CUH

#include <stdint.h>
#include <stdlib.h>
#include <string.h>

#if defined(MEEP_HOST_ONLY) || !defined(__CUDACC__)
  #define MHD
  #define MDEV
#else
  #define MHD __host__ __device__
  #define MDEV __device__
#endif

// ------------------------------------------------------------------ frozen v2 constants -------
#define MEEP_ALGO_VERSION 0        /* meep_xof frames this byte (see blake3_xof.hpp) */
#define MEEP_PARAM_PIPE   60       /* param-set id used by v2_ctx_create (v1_config(60,...)) */
#define MEEP_PARAM_DS     0        /* param-set id used by the v2 dataset seed XOF */
#define V2_SEED_WORDS     8192u
#define V2_NPARENTS       4
#define DATASET_WORDS     0x400000u   /* 32 MiB */
#define SCRATCH_WORDS     0x100000u   /* 8 MiB  */
#define PROG_LEN          256u
#define ROUNDS            400u
#define STEPS_PER_ROUND   256u
#define S3_STRIDE         64u
#define MIXBACK_INTERVAL  8u

// ------------------------------------------------------------------------ BLAKE3 -------------
#define B3_CHUNK_START 1u
#define B3_CHUNK_END   2u
#define B3_ROOT        8u
#define B3_DERIVE_KEY_CONTEXT  32u
#define B3_DERIVE_KEY_MATERIAL 64u

MHD inline uint32_t b3_rotr(uint32_t x, int n) { return (x >> n) | (x << (32 - n)); }

MHD inline void b3_g(uint32_t* s, int a, int b, int c, int d, uint32_t mx, uint32_t my) {
    s[a] = s[a] + s[b] + mx;  s[d] = b3_rotr(s[d] ^ s[a], 16);
    s[c] = s[c] + s[d];       s[b] = b3_rotr(s[b] ^ s[c], 12);
    s[a] = s[a] + s[b] + my;  s[d] = b3_rotr(s[d] ^ s[a], 8);
    s[c] = s[c] + s[d];       s[b] = b3_rotr(s[b] ^ s[c], 7);
}

MHD inline void b3_round(uint32_t* s, const uint32_t* m) {
    b3_g(s, 0, 4, 8, 12, m[0], m[1]);   b3_g(s, 1, 5, 9, 13, m[2], m[3]);
    b3_g(s, 2, 6, 10, 14, m[4], m[5]);  b3_g(s, 3, 7, 11, 15, m[6], m[7]);
    b3_g(s, 0, 5, 10, 15, m[8], m[9]);  b3_g(s, 1, 6, 11, 12, m[10], m[11]);
    b3_g(s, 2, 7, 8, 13, m[12], m[13]); b3_g(s, 3, 4, 9, 14, m[14], m[15]);
}

MHD inline void b3_permute(uint32_t* m) {
    const int P[16] = {2, 6, 3, 10, 7, 0, 4, 13, 1, 11, 12, 5, 9, 14, 15, 8};
    uint32_t t[16];
    for (int i = 0; i < 16; ++i) t[i] = m[P[i]];
    for (int i = 0; i < 16; ++i) m[i] = t[i];
}

// Full 16-word compression output (state after feed-forward).
MHD inline void b3_compress(const uint32_t cv[8], const uint32_t block[16], uint64_t counter,
                            uint32_t block_len, uint32_t flags, uint32_t out[16]) {
    const uint32_t IV[8] = {0x6A09E667u, 0xBB67AE85u, 0x3C6EF372u, 0xA54FF53Au,
                            0x510E527Fu, 0x9B05688Cu, 0x1F83D9ABu, 0x5BE0CD19u};
    uint32_t s[16] = {cv[0], cv[1], cv[2], cv[3], cv[4], cv[5], cv[6], cv[7],
                      IV[0], IV[1], IV[2], IV[3],
                      (uint32_t)(counter & 0xFFFFFFFFu), (uint32_t)(counter >> 32), block_len, flags};
    uint32_t m[16];
    for (int i = 0; i < 16; ++i) m[i] = block[i];
    for (int r = 0; r < 7; ++r) { b3_round(s, m); if (r < 6) b3_permute(m); }
    for (int i = 0; i < 8; ++i) { s[i] ^= s[i + 8]; s[i + 8] ^= cv[i]; }
    for (int i = 0; i < 16; ++i) out[i] = s[i];
}

MHD inline void b3_words_from_le(const uint8_t* p, uint32_t* w, int n) {
    for (int i = 0; i < n; ++i)
        w[i] = (uint32_t)p[4 * i] | ((uint32_t)p[4 * i + 1] << 8) |
               ((uint32_t)p[4 * i + 2] << 16) | ((uint32_t)p[4 * i + 3] << 24);
}

// Single-chunk (len <= 1024) keyed hash + XOF. Covers every meep_xof call in frozen v2.
MHD inline void b3_xof_1chunk(const uint32_t key[8], const uint8_t* in, uint32_t len,
                              uint32_t base_flags, uint8_t* out, uint32_t outlen) {
    uint32_t cv[8];
    for (int i = 0; i < 8; ++i) cv[i] = key[i];
    uint32_t nblocks = (len + 63) / 64; if (nblocks == 0) nblocks = 1;
    uint32_t last_block[16];
    uint32_t last_len = 0, last_flags = 0;
    for (uint32_t b = 0; b < nblocks; ++b) {
        uint8_t buf[64];
        uint32_t blen = (b + 1 == nblocks) ? (len - b * 64) : 64;
        if (len == 0) blen = 0;
        for (int i = 0; i < 64; ++i) buf[i] = (i < (int)blen) ? in[b * 64 + i] : 0;
        uint32_t blk[16];
        b3_words_from_le(buf, blk, 16);
        uint32_t fl = base_flags | (b == 0 ? B3_CHUNK_START : 0u) |
                      (b + 1 == nblocks ? B3_CHUNK_END : 0u);
        if (b + 1 == nblocks) {
            for (int i = 0; i < 16; ++i) last_block[i] = blk[i];
            last_len = blen; last_flags = fl | B3_ROOT;
        } else {
            uint32_t st[16];
            b3_compress(cv, blk, 0, 64, fl, st);
            for (int i = 0; i < 8; ++i) cv[i] = st[i];
        }
    }
    uint32_t produced = 0, ctr = 0;
    while (produced < outlen) {
        uint32_t st[16];
        b3_compress(cv, last_block, ctr, last_len, last_flags, st);
        for (int w = 0; w < 16 && produced < outlen; ++w)
            for (int b = 0; b < 4 && produced < outlen; ++b)
                out[produced++] = (uint8_t)((st[w] >> (8 * b)) & 0xFF);
        ++ctr;
    }
}

// Derive-key context key: hash the context string with key=IV, flags=DERIVE_KEY_CONTEXT.
MHD inline void b3_context_key(const char* ctx, uint32_t len, uint32_t out_key[8]) {
    const uint32_t IV[8] = {0x6A09E667u, 0xBB67AE85u, 0x3C6EF372u, 0xA54FF53Au,
                            0x510E527Fu, 0x9B05688Cu, 0x1F83D9ABu, 0x5BE0CD19u};
    uint8_t out32[32];
    b3_xof_1chunk(IV, (const uint8_t*)ctx, len, B3_DERIVE_KEY_CONTEXT, out32, 32);
    b3_words_from_le(out32, out_key, 8);
}

// ---------------------------------------------------------------- framed meep_xof ------------
struct MField { const uint8_t* p; uint32_t n; };

// Builds ALGO_VERSION || PARAM_ID || (LE64(len)||bytes)* into `buf`, then keyed XOF.
MHD inline void meep_xof(const uint32_t ctx_key[8], uint8_t param_id, const MField* f, int nf,
                         uint8_t* buf, uint8_t* out, uint32_t outlen) {
    uint32_t n = 0;
    buf[n++] = (uint8_t)MEEP_ALGO_VERSION;
    buf[n++] = param_id;
    for (int i = 0; i < nf; ++i) {
        uint64_t L = f[i].n;
        for (int b = 0; b < 8; ++b) buf[n++] = (uint8_t)((L >> (8 * b)) & 0xFF);
        for (uint32_t k = 0; k < f[i].n; ++k) buf[n++] = f[i].p[k];
    }
    b3_xof_1chunk(ctx_key, buf, n, B3_DERIVE_KEY_MATERIAL, out, outlen);
}

// --------------------------------------------------------------- 64-bit helpers --------------
MHD inline uint64_t rotl64d(uint64_t x, unsigned n) { n &= 63; return n ? ((x << n) | (x >> (64 - n))) : x; }
MHD inline uint64_t rotr64d(uint64_t x, unsigned n) { n &= 63; return n ? ((x >> n) | (x << (64 - n))) : x; }
MHD inline uint64_t mulhi64d(uint64_t a, uint64_t b) {
#if defined(__CUDA_ARCH__)
    return __umul64hi(a, b);
#else
    uint64_t aL = a & 0xffffffffULL, aH = a >> 32, bL = b & 0xffffffffULL, bH = b >> 32;
    uint64_t ll = aL * bL, lh = aL * bH, hl = aH * bL, hh = aH * bH;
    uint64_t cross = (ll >> 32) + (lh & 0xffffffffULL) + (hl & 0xffffffffULL);
    return hh + (lh >> 32) + (hl >> 32) + (cross >> 32);
#endif
}
MHD inline uint64_t ld64le(const uint8_t* p) {
    uint64_t v = 0;
    for (int i = 0; i < 8; ++i) v |= ((uint64_t)p[i]) << (8 * i);
    return v;
}
MHD inline void st64le(uint8_t* p, uint64_t v) { for (int i = 0; i < 8; ++i) p[i] = (uint8_t)(v >> (8 * i)); }
MHD inline void st32le(uint8_t* p, uint32_t v) { for (int i = 0; i < 4; ++i) p[i] = (uint8_t)(v >> (8 * i)); }
MHD inline uint64_t byteshufd(uint64_t x, uint64_t sel) {
    uint8_t in[8], out[8];
    st64le(in, x);
    for (int i = 0; i < 8; ++i) out[i] = in[(sel >> (3 * i)) & 7u];
    return ld64le(out);
}

// ---------------------------------------------------------------- v2 dataset ------------------
MHD inline uint64_t v2_splitmixd(uint64_t z) {
    z += 0x9E3779B97F4A7C15ULL;
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
MHD inline uint64_t v2_mixd(uint64_t a, uint64_t b, uint64_t w) {
    uint64_t x = a + b;
    x ^= rotl64d(x, 29);
    x = x * (b | 1ULL);
    x ^= w;
    x = rotl64d(x, 17);
    return x + a;
}

// ---------------------------------------------------------------- opcodes ---------------------
enum { OP_ADD64 = 0, OP_XOR64, OP_MUL64, OP_MULHI64, OP_ROTL64, OP_ROTR64, OP_ADD32, OP_XOR32,
       OP_MUL32, OP_LOAD64_SCRATCH, OP_LOAD64_DATASET, OP_STORE64, OP_BRANCH_IF_BIT, OP_CSELECT,
       OP_BYTE_SHUFFLE };
// Run-length weights (must sum to 256), identical to params.hpp.
MHD inline void build_opcode_table(uint8_t t[256]) {
    const uint8_t ops[15] = {OP_ADD64, OP_XOR64, OP_MUL64, OP_MULHI64, OP_ROTL64, OP_ROTR64,
                             OP_ADD32, OP_XOR32, OP_MUL32, OP_LOAD64_SCRATCH, OP_LOAD64_DATASET,
                             OP_STORE64, OP_BRANCH_IF_BIT, OP_CSELECT, OP_BYTE_SHUFFLE};
    const uint16_t cnt[15] = {41, 28, 20, 12, 8, 7, 16, 12, 10, 31, 15, 20, 18, 9, 9};
    int idx = 0;
    for (int i = 0; i < 15; ++i)
        for (uint16_t c = 0; c < cnt[i]; ++c) t[idx++] = ops[i];
}
struct MInstr { uint8_t op, d, s, a; uint32_t imm; };

// ---------------------------------------------------------------- per-nonce pipeline ----------
// Context keys, precomputed once (host) and passed to the device.
struct MeepKeys {
    uint32_t k_dataset[8], k_program[8], k_nonce[8], k_scratchseed[8], k_final[8];
};
MHD inline void meep_init_keys(MeepKeys* K) {
    b3_context_key("MEEP/DATASET/v2", 15, K->k_dataset);
    b3_context_key("MEEP/PROGRAM/v1", 15, K->k_program);
    b3_context_key("MEEP/NONCE/v1", 13, K->k_nonce);
    b3_context_key("MEEP/SCRATCHSEED/v1", 19, K->k_scratchseed);
    b3_context_key("MEEP/FINAL/v1", 13, K->k_final);
}

// Derive the per-job program (constant across nonces).
MHD inline void meep_derive_program(const MeepKeys* K, const uint8_t epochKey[32],
                                    const uint8_t seedHash[32], uint64_t height,
                                    MInstr* prog, uint8_t* scratchbuf /*>=2048*/,
                                    uint8_t* framebuf /*>=128*/) {
    uint8_t h_le[8];
    st64le(h_le, height);
    MField f[3] = {{h_le, 8}, {epochKey, 32}, {seedHash, 32}};
    meep_xof(K->k_program, MEEP_PARAM_PIPE, f, 3, framebuf, scratchbuf, PROG_LEN * 8);
    uint8_t tab[256];
    build_opcode_table(tab);
    for (uint32_t i = 0; i < PROG_LEN; ++i) {
        const uint8_t* p = scratchbuf + i * 8;
        prog[i].op = tab[p[0]];
        prog[i].d = p[1] & 7u;
        prog[i].s = p[2] & 7u;
        prog[i].a = p[3] & 63u;
        prog[i].imm = (uint32_t)p[4] | ((uint32_t)p[5] << 8) | ((uint32_t)p[6] << 16) | ((uint32_t)p[7] << 24);
    }
}

// Scratchpad addressing: `sp_stride` selects the layout.
//   contiguous per-thread : base = pool + tid*SCRATCH_WORDS, sp_stride = 1
//   interleaved (coalesced): base = pool + tid,              sp_stride = nthreads
// One implementation serves both GPU mappings, so exactness is identical in each.
#define SP_AT(SP, i) (SP)[(uint64_t)(i) * sp_stride]

// Full per-nonce hash. `SP` must provide SCRATCH_WORDS words for this nonce.
MHD inline void meep_v2_hash_one(const MeepKeys* K, const uint64_t* dataset, const MInstr* prog,
                                 const uint8_t* tmpl, uint32_t tmpl_len, uint32_t nonce,
                                 uint64_t* SP, uint64_t sp_stride, uint8_t* framebuf /*>=512*/,
                                 uint8_t* out32) {
    // 1) per-nonce seed (96 bytes)
    uint8_t seed[96];
    {
        uint8_t n_le[4];
        st32le(n_le, nonce);
        MField f[2] = {{n_le, 4}, {tmpl, tmpl_len}};
        meep_xof(K->k_nonce, MEEP_PARAM_PIPE, f, 2, framebuf, seed, 96);
    }
    // 2) S3 scratchpad init: sparse BLAKE3 checkpoints + dependent expansion
    {
        const uint32_t nck = SCRATCH_WORDS / S3_STRIDE;
        uint8_t n_le[4];
        st32le(n_le, nonce);
        MField f[2] = {{n_le, 4}, {seed, 96}};
        // checkpoints are produced in chunks to bound temp memory: XOF is deterministic over the
        // whole stream, so we generate it in 64-byte blocks directly into the scratchpad slots.
        // (Equivalent to one XOF of nck*8 bytes.)
        uint8_t ckbuf[64];
        uint32_t produced = 0;
        // Build the framed input once, then stream XOF blocks.
        uint32_t n = 0;
        framebuf[n++] = (uint8_t)MEEP_ALGO_VERSION;
        framebuf[n++] = MEEP_PARAM_PIPE;
        for (int i = 0; i < 2; ++i) {
            uint64_t L = f[i].n;
            for (int b = 0; b < 8; ++b) framebuf[n++] = (uint8_t)((L >> (8 * b)) & 0xFF);
            for (uint32_t k = 0; k < f[i].n; ++k) framebuf[n++] = f[i].p[k];
        }
        // single-chunk XOF, streaming 64 bytes at a time
        uint32_t cv[8];
        for (int i = 0; i < 8; ++i) cv[i] = K->k_scratchseed[i];
        uint32_t nblocks = (n + 63) / 64; if (nblocks == 0) nblocks = 1;
        uint32_t last_block[16], last_len = 0, last_flags = 0;
        for (uint32_t b = 0; b < nblocks; ++b) {
            uint8_t bb[64];
            uint32_t blen = (b + 1 == nblocks) ? (n - b * 64) : 64;
            for (int i = 0; i < 64; ++i) bb[i] = (i < (int)blen) ? framebuf[b * 64 + i] : 0;
            uint32_t blk[16];
            b3_words_from_le(bb, blk, 16);
            uint32_t fl = B3_DERIVE_KEY_MATERIAL | (b == 0 ? B3_CHUNK_START : 0u) |
                          (b + 1 == nblocks ? B3_CHUNK_END : 0u);
            if (b + 1 == nblocks) {
                for (int i = 0; i < 16; ++i) last_block[i] = blk[i];
                last_len = blen; last_flags = fl | B3_ROOT;
            } else {
                uint32_t st[16];
                b3_compress(cv, blk, 0, 64, fl, st);
                for (int i = 0; i < 8; ++i) cv[i] = st[i];
            }
        }
        uint32_t ctr = 0, ck = 0;
        while (ck < nck) {
            uint32_t st[16];
            b3_compress(cv, last_block, ctr, last_len, last_flags, st);
            for (int w = 0; w < 16; ++w) st32le(ckbuf + w * 4, st[w]);
            for (int q = 0; q < 8 && ck < nck; ++q, ++ck)
                SP_AT(SP, (uint64_t)ck * S3_STRIDE) = ld64le(ckbuf + q * 8);
            ++ctr;
            produced += 64;
        }
        (void)produced;
        for (uint32_t w = 0; w < SCRATCH_WORDS; ++w) {
            if (w % S3_STRIDE == 0) continue;
            uint64_t prev = SP_AT(SP, w - 1);
            uint64_t anchor = SP_AT(SP, (w / S3_STRIDE) * S3_STRIDE);
            SP_AT(SP, w) = v2_mixd(prev, anchor, (uint64_t)w);
        }
    }
    // 3) VM: ROUNDS x STEPS with the mandatory dataset read-chain and mix-back
    uint64_t r[8], acc[4], last[8], spos = 0;
    for (int i = 0; i < 8; ++i) r[i] = ld64le(seed + i * 8);
    for (int i = 0; i < 4; ++i) acc[i] = ld64le(seed + 64 + i * 8);
    for (int i = 0; i < 8; ++i) last[i] = 0;
    const uint64_t smask = SCRATCH_WORDS - 1, dmask = DATASET_WORDS - 1;
    uint64_t t = 0;
    for (uint32_t round = 0; round < ROUNDS; ++round) {
        uint32_t pc = 0;
        for (uint32_t step = 0; step < STEPS_PER_ROUND; ++step) {
            MInstr I = prog[pc];
            uint64_t d = I.d, s = I.s, a = I.a, imm = I.imm;
            uint64_t addr = 0, val = 0;
            bool mem = false, taken = false;
            switch (I.op) {
                case OP_ADD64: r[d] = r[d] + r[s] + imm; break;
                case OP_XOR64: r[d] = r[d] ^ r[s] ^ imm; break;
                case OP_MUL64: r[d] = r[d] * (r[s] | 1ULL); break;
                case OP_MULHI64: r[d] = mulhi64d(r[d], r[s]); break;
                case OP_ROTL64: r[d] = rotl64d(r[d], (unsigned)((r[s] + imm) & 63)); break;
                case OP_ROTR64: r[d] = rotr64d(r[d], (unsigned)((r[s] + imm) & 63)); break;
                case OP_ADD32: r[d] = (uint64_t)(uint32_t)((uint32_t)r[d] + (uint32_t)r[s] + (uint32_t)imm); break;
                case OP_XOR32: r[d] = (uint64_t)(uint32_t)((uint32_t)r[d] ^ (uint32_t)r[s] ^ (uint32_t)imm); break;
                case OP_MUL32: r[d] = (uint64_t)(uint32_t)((uint32_t)r[d] * ((uint32_t)r[s] | 1u)); break;
                case OP_LOAD64_SCRATCH: {
                    uint64_t off = (r[s] + imm) & smask;
                    if (((acc[1] ^ r[s]) & 3u) == 0) off = last[(acc[1] >> 2) & 7u] & smask;
                    val = SP_AT(SP, off); r[d] = val; addr = off; mem = true;
                } break;
                case OP_LOAD64_DATASET: {
                    uint64_t off = (r[s] ^ imm ^ acc[0]) & dmask;
                    val = dataset[off]; r[d] = val; addr = off; mem = true;
                } break;
                case OP_STORE64: {
                    uint64_t off = (r[d] + imm) & smask;
                    val = r[s];
                    SP_AT(SP, off) = val;
                    last[spos & 7u] = off; spos++;
                    addr = off; mem = true;
                } break;
                case OP_BRANCH_IF_BIT: taken = ((r[s] >> a) & 1u) != 0; break;
                case OP_CSELECT: r[d] = (r[s] & 1u) ? (r[d] + imm) : (r[d] ^ r[s]); break;
                case OP_BYTE_SHUFFLE: r[d] = byteshufd(r[d], r[s]); break;
                default: break;
            }
            acc[0] = rotl64d(acc[0] + (uint64_t)I.op + imm, 1);
            acc[1] = acc[1] ^ r[d] ^ (((uint64_t)d << 3) | (uint64_t)s);
            acc[2] = acc[2] * (r[s] | 1ULL) + acc[0];
            if (mem) acc[3] = rotl64d(acc[3] ^ addr, 17) + val; else acc[3] = acc[3] + acc[2];
            // mandatory data-dependent dataset read-chain
            uint64_t da = (acc[2] ^ r[step & 7]) & dmask;
            uint64_t dv = dataset[da];
            acc[3] ^= dv;
            acc[2] = rotl64d(acc[2] + dv, 23);
            ++t;
            if ((t % MIXBACK_INTERVAL) == 0) {
                unsigned j = (unsigned)((t / MIXBACK_INTERVAL) & 7u);
                r[j] ^= SP_AT(SP, last[j] & smask);
            }
            if (taken) pc = (uint32_t)((pc + 1 + (I.imm & (PROG_LEN - 1))) % PROG_LEN);
            else pc = (pc + 1) % PROG_LEN;
        }
    }
    // 4) final walk + finalization
    uint64_t w32[32];
    for (int k = 0; k < 8; ++k) w32[k] = SP_AT(SP, last[k] & smask);
    for (int k = 0; k < 24; ++k) {
        uint64_t idx = (acc[k & 3] ^ r[k & 7] ^ (uint64_t)k * 0x9E3779B97F4A7C15ULL) & smask;
        w32[8 + k] = SP_AT(SP, idx);
    }
    acc[0] ^= w32[0]; acc[1] += w32[8]; acc[2] ^= w32[16]; acc[3] += w32[24];
    uint8_t rb[64], ab[32], sample[256];
    for (int i = 0; i < 8; ++i) st64le(rb + i * 8, r[i]);
    for (int i = 0; i < 4; ++i) st64le(ab + i * 8, acc[i]);
    for (int k = 0; k < 32; ++k) st64le(sample + k * 8, w32[k]);
    MField f3[3] = {{rb, 64}, {ab, 32}, {sample, 256}};
    meep_xof(K->k_final, MEEP_PARAM_PIPE, f3, 3, framebuf, out32, 32);
}

// ------------------------------------------------------------ host-side dataset build ---------
// HOST-ONLY BY ANNOTATION, NOT BY PREPROCESSOR. This function has no MHD/__device__ marker, so nvcc
// generates code for it only on the host — exactly what we want (it uses malloc/free and runs once
// per epoch before upload).
//
// Do NOT wrap it in a preprocessor guard. Two guards were tried and both were wrong:
//   #if !defined(__CUDACC__)   -> nvcc defines __CUDACC__ in BOTH its host and device passes, so
//                                 the function vanished from every nvcc build.
//   #if !defined(__CUDA_ARCH__)-> __CUDA_ARCH__ is only defined in the device pass, but nvcc still
//                                 PARSES the whole translation unit (including main()) during that
//                                 pass, so main()'s call site saw an undefined identifier.
// Leaving it unguarded is both simpler and correct: visible to every pass, compiled only for host.
inline void meep_v2_build_dataset(const MeepKeys* K, const uint8_t epochKey[32], uint64_t* D) {
    // seed region via framed XOF (field order: LE64(0), epochKey), param id 0
    static uint8_t framebuf[128];
    uint8_t idx0[8];
    st64le(idx0, 0);
    MField f[2] = {{idx0, 8}, {epochKey, 32}};
    uint8_t* tmp = (uint8_t*)malloc((size_t)V2_SEED_WORDS * 8);
    meep_xof(K->k_dataset, MEEP_PARAM_DS, f, 2, framebuf, tmp, V2_SEED_WORDS * 8);
    for (uint32_t w = 0; w < V2_SEED_WORDS; ++w) D[w] = ld64le(tmp + w * 8);
    free(tmp);
    uint64_t seedconst = D[0];
    for (uint64_t w = V2_SEED_WORDS; w < DATASET_WORDS; ++w) {
        uint64_t h = v2_splitmixd(w ^ seedconst);
        uint64_t p1 = h % w;              uint64_t d1 = D[p1];
        uint64_t acc = v2_mixd(d1, h, w);
        uint64_t p2 = (d1 ^ (h >> 13)) % w; uint64_t d2 = D[p2];
        acc = v2_mixd(acc, d2, w ^ rotl64d(d1, 32));
        uint64_t p3 = (d1 ^ d2 ^ (h >> 29)) % w; uint64_t d3 = D[p3];
        acc = v2_mixd(acc, d3, w ^ rotl64d(d2, 17));
        uint64_t p4 = (d2 ^ d3 ^ (h >> 7)) % w; uint64_t d4 = D[p4];
        D[w] = v2_mixd(acc, d4, w ^ rotl64d(d3, 41));
    }
}

#endif  // MEEP_V2_CUDA_CORE_CUH


In [ ]:
%%writefile vectors_v2.txt
v2 4 0 3b0ff27e098298e936d297e7bbf41d1dbe491db0f2d405c7818a12c4af561a61
v2 4 1 9e87f1cf8e65a514ece04bd816cedea417ce83bfb5137b87fccd6e4cc7a47ec1
v2 4 2 97581b8f03f203e39e29191958f38d670adf22ffd5e1dd516fde1db71e6107c2
v2 4 3 b1d798ee4a1b0ce5d817433839b97011cfe931809228c61425bbc1ca1d23d1dc
v2 4 4 a43accb406917f738c0db6111c7327b6d680c77a9f885e0a7fb7b95b39127df2
v2 4 5 f4772ec91bc6f944838221258875c259ba4416e39953ba4b19d2057e53618d6d
v2 4 6 e5572b7903c4988a328c3214cf5f828b55d0637f5cd2dd03e50e6800b83b66b9
v2 4 7 890c5a0ebbd9b7b2147f28ef52df961e96b875e01a52ed9ec53669725cc2ec94
v2 4 8 5a3c780773b12647be59704d2f57bb00118668d509f3c42466255684b5d0ac79
v2 4 9 7b5ab8ac031cbec2c769d7bb21e3f853d0742676a7fbeb61e20f65fb9d923045
v2 4 10 ef0e3463e43d5bd3165a640b88e32c76d9dee519a59b1deb0aaabe405a54dab6
v2 4 11 33b5159dd95909e3262634a9c81fbdfcb0b3043a2d454fa6ee3b8d89213f60c6
v2 4 12 d13fedb322ab49bf32320f842bd3a93134b1c38839476bf956ad3dfccd8cecc9
v2 4 13 f0b4c56094f9cf868eeaf477e245b1a75d6fcad6a32ce535194a17d43ea6142a
v2 4 14 46fe0d0014125e82dadca129bfc44b1aede35ca03f0f3c9d6490209dd8ec4d24
v2 4 15 92c4ac1ee6488429a94a4da91bc5a793daa897e2f57fc626cfa61966d29d810e
v2 4 16 ce697bd4784e857611e5e6ebe6c01c4289c69ac98259879fffad9768c38f4e9a
v2 4 17 4c1c23290c57762d4eb6737b457cc4ddb863c7e7f07f05d1857f22fa4eb551f9
v2 4 18 1fa1564de8e04bbea578ae5d6fe0e055fe77729d7d4d54dfb088c4dc99704d92
v2 4 19 c349e5550a19da32c4005a5228a1470b707f63d3963399e13997b6bc010b4ba0


In [ ]:
%%writefile v2_gpu.cu
#include "v2_cuda_core.cuh"
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

#define CK(x) do { cudaError_t e=(x); if(e!=cudaSuccess){ printf("CUDA_ERR %s @%d\n", cudaGetErrorString(e), __LINE__); exit(1);} } while(0)

// Mapping A: one thread per nonce, CONTIGUOUS per-thread scratchpad (sp_stride = 1).
__global__ void k_contig(const MeepKeys* K, const uint64_t* ds, const MInstr* prog,
                         const uint8_t* tmpl, uint32_t tlen, uint32_t base_nonce, int n,
                         uint64_t* pool, uint8_t* outs) {
    int i = blockIdx.x * blockDim.x + threadIdx.x; if (i >= n) return;
    uint8_t fb[512];
    meep_v2_hash_one(K, ds, prog, tmpl, tlen, base_nonce + i,
                     pool + (uint64_t)i * SCRATCH_WORDS, 1, fb, outs + (uint64_t)i * 32);
}

// Mapping B: one thread per nonce, INTERLEAVED scratchpad (sp_stride = n) so concurrent threads'
// word accesses are adjacent -> coalesced global memory transactions.
__global__ void k_interleaved(const MeepKeys* K, const uint64_t* ds, const MInstr* prog,
                              const uint8_t* tmpl, uint32_t tlen, uint32_t base_nonce, int n,
                              uint64_t* pool, uint8_t* outs) {
    int i = blockIdx.x * blockDim.x + threadIdx.x; if (i >= n) return;
    uint8_t fb[512];
    meep_v2_hash_one(K, ds, prog, tmpl, tlen, base_nonce + i,
                     pool + i, (uint64_t)n, fb, outs + (uint64_t)i * 32);
}

// modes: verify | sweep | steady   (occupancy lives in v2_occ.cu, a separate TU)
int main(int argc, char** argv) {
    const char* mode = argc > 1 ? argv[1] : "sweep";
    int batch   = argc > 2 ? atoi(argv[2]) : 1536;
    int mapping = argc > 3 ? atoi(argv[3]) : 1;     // 0 contiguous, 1 interleaved
    int tpb     = argc > 4 ? atoi(argv[4]) : 32;
    double secs = argc > 5 ? atof(argv[5]) : 30.0;

    MeepKeys hK; meep_init_keys(&hK);
    uint8_t ek[32], sh[32];
    for (int i = 0; i < 32; ++i) { ek[i] = (uint8_t)(i*7+1); sh[i] = (uint8_t)(i*3+9); }
    uint8_t tmpl[8] = {1,2,3,4,5,6,7,8};

    // dataset + program built on the HOST once (as a real miner would per epoch/job), then uploaded
    std::vector<uint64_t> D(DATASET_WORDS);
    meep_v2_build_dataset(&hK, ek, D.data());
    std::vector<MInstr> prog(PROG_LEN);
    { std::vector<uint8_t> pb(PROG_LEN*8); uint8_t fb[256];
      meep_derive_program(&hK, ek, sh, 4096, prog.data(), pb.data(), fb); }

    MeepKeys* dK; uint64_t* dD; MInstr* dP; uint8_t* dT; uint8_t* dOut; uint64_t* dPool;
    CK(cudaMalloc(&dK,sizeof(MeepKeys)));   CK(cudaMemcpy(dK,&hK,sizeof(MeepKeys),cudaMemcpyHostToDevice));
    CK(cudaMalloc(&dD,DATASET_WORDS*8));    CK(cudaMemcpy(dD,D.data(),DATASET_WORDS*8,cudaMemcpyHostToDevice));
    CK(cudaMalloc(&dP,PROG_LEN*sizeof(MInstr))); CK(cudaMemcpy(dP,prog.data(),PROG_LEN*sizeof(MInstr),cudaMemcpyHostToDevice));
    CK(cudaMalloc(&dT,8));                  CK(cudaMemcpy(dT,tmpl,8,cudaMemcpyHostToDevice));
    CK(cudaMalloc(&dOut,(size_t)batch*32));

    size_t poolBytes = (size_t)batch * SCRATCH_WORDS * 8;
    size_t freeB=0, totB=0; cudaMemGetInfo(&freeB,&totB);
    if (cudaMalloc(&dPool, poolBytes) != cudaSuccess) {
        printf("ALLOC_FAIL batch=%d needed_GiB=%.2f free_GiB=%.2f\n",
               batch, poolBytes/1073741824.0, freeB/1073741824.0);
        return 2;
    }
    int blocks = (batch + tpb - 1) / tpb;
    auto launch = [&](uint32_t base) {
        if (mapping == 0) k_contig<<<blocks,tpb>>>(dK,dD,dP,dT,8,base,batch,dPool,dOut);
        else              k_interleaved<<<blocks,tpb>>>(dK,dD,dP,dT,8,base,batch,dPool,dOut);
    };

    if (!strcmp(mode, "verify")) {
        launch(0); CK(cudaDeviceSynchronize());
        std::vector<uint8_t> h((size_t)batch*32);
        CK(cudaMemcpy(h.data(),dOut,(size_t)batch*32,cudaMemcpyDeviceToHost));
        for (int i = 0; i < batch; ++i) {
            printf("gpu %d ", i);
            for (int b = 0; b < 32; ++b) printf("%02x", h[(size_t)i*32+b]);
            printf("\n");
        }
        return 0;
    }

    // warmup (excluded from all timing and from the power window)
    launch(0); CK(cudaDeviceSynchronize());
    size_t usedB=0; { size_t f,t; cudaMemGetInfo(&f,&t); usedB = t - f; }

    if (!strcmp(mode, "sweep")) {                 // short throughput probe for the sweeps
        auto t0 = std::chrono::steady_clock::now();
        int waves = 3;
        for (int w = 0; w < waves; ++w) launch((uint32_t)w*batch);
        CK(cudaDeviceSynchronize());
        double s = std::chrono::duration<double>(std::chrono::steady_clock::now()-t0).count();
        printf("SWEEP mapping=%d batch=%d tpb=%d hps=%.2f vram_gib=%.2f\n",
               mapping, batch, tpb, (double)batch*waves/s, usedB/1073741824.0);
        return 0;
    }

    // steady: continuous hashing for `secs`; prints a machine-readable window for the power poller
    printf("STEADY_BEGIN\n"); fflush(stdout);
    auto t0 = std::chrono::steady_clock::now();
    unsigned long long done = 0; uint32_t base = 0;
    while (std::chrono::duration<double>(std::chrono::steady_clock::now()-t0).count() < secs) {
        launch(base); base += batch;
        CK(cudaDeviceSynchronize());
        done += batch;
    }
    double s = std::chrono::duration<double>(std::chrono::steady_clock::now()-t0).count();
    printf("STEADY_END\n");
    printf("STEADY mapping=%d batch=%d tpb=%d secs=%.2f hashes=%llu hps=%.2f vram_gib=%.2f\n",
           mapping, batch, tpb, s, done, done/s, usedB/1073741824.0);
    return 0;
}


In [ ]:
%%writefile v2_occ.cu
// Standalone occupancy report. Kept in its OWN translation unit so that if any CUDA occupancy API
// detail differs by toolkit version, the main benchmark still builds and runs.
#include "v2_cuda_core.cuh"
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

__global__ void k_contig(const MeepKeys* K, const uint64_t* ds, const MInstr* prog,
                         const uint8_t* tmpl, uint32_t tlen, uint32_t base_nonce, int n,
                         uint64_t* pool, uint8_t* outs) {
    int i = blockIdx.x * blockDim.x + threadIdx.x; if (i >= n) return;
    uint8_t fb[512];
    meep_v2_hash_one(K, ds, prog, tmpl, tlen, base_nonce + i,
                     pool + (uint64_t)i * SCRATCH_WORDS, 1, fb, outs + (uint64_t)i * 32);
}
__global__ void k_interleaved(const MeepKeys* K, const uint64_t* ds, const MInstr* prog,
                              const uint8_t* tmpl, uint32_t tlen, uint32_t base_nonce, int n,
                              uint64_t* pool, uint8_t* outs) {
    int i = blockIdx.x * blockDim.x + threadIdx.x; if (i >= n) return;
    uint8_t fb[512];
    meep_v2_hash_one(K, ds, prog, tmpl, tlen, base_nonce + i,
                     pool + i, (uint64_t)n, fb, outs + (uint64_t)i * 32);
}

int main(int argc, char** argv) {
    int tpb = argc > 1 ? atoi(argv[1]) : 32;
    cudaDeviceProp p;
    if (cudaGetDeviceProperties(&p, 0) != cudaSuccess) { printf("no device\n"); return 1; }
    int warpsMax = p.maxThreadsPerMultiProcessor / p.warpSize;
    printf("device=%s SMs=%d maxThreadsPerSM=%d warpSize=%d regsPerSM=%d\n",
           p.name, p.multiProcessorCount, p.maxThreadsPerMultiProcessor, p.warpSize,
           p.regsPerMultiprocessor);
    const void* ks[2] = {(const void*)k_contig, (const void*)k_interleaved};
    const char* nm[2] = {"contiguous", "interleaved"};
    for (int i = 0; i < 2; ++i) {
        int nb = 0;
        cudaFuncAttributes a;
        cudaError_t e1 = cudaOccupancyMaxActiveBlocksPerMultiprocessor(&nb, ks[i], tpb, 0);
        cudaError_t e2 = cudaFuncGetAttributes(&a, ks[i]);
        if (e1 != cudaSuccess || e2 != cudaSuccess) { printf("%s: query failed\n", nm[i]); continue; }
        int activeWarps = nb * tpb / p.warpSize;
        printf("%-12s tpb=%d blocks/SM=%d activeWarps/SM=%d theoWarps/SM=%d occupancy=%.3f "
               "regs/thread=%d localMem/thread=%zu shared/block=%zu\n",
               nm[i], tpb, nb, activeWarps, warpsMax, (double)activeWarps / warpsMax,
               a.numRegs, (size_t)a.localSizeBytes, (size_t)a.sharedSizeBytes);
    }
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 -o v2_gpu v2_gpu.cu && echo BUILD_OK
# occupancy reporter is a SEPARATE translation unit: if it fails to build, the benchmark
# above still runs (only the occupancy numbers are lost).
!nvcc -O3 -arch=sm_75 -o v2_occ v2_occ.cu && echo OCC_BUILD_OK || echo OCC_BUILD_FAILED_nonfatal


## 1) Correctness - must be byte-identical to the committed vectors
Both mappings, nonces 0..19, diffed against `vectors_v2.txt`. Performance below is only valid if this passes.


In [ ]:
import subprocess
exp = {}
for line in open('vectors_v2.txt'):
    p = line.split()
    if len(p)==4 and p[0]=='v2': exp[int(p[2])] = p[3]
print('expected vectors:', len(exp))
CORRECT = True
for mapping,name in [(0,'contiguous'),(1,'interleaved')]:
    out = subprocess.run(['./v2_gpu','verify','20',str(mapping),'32'],capture_output=True,text=True).stdout
    got = {}
    for line in out.splitlines():
        p=line.split()
        if len(p)==3 and p[0]=='gpu': got[int(p[1])]=p[2]
    bad = [n for n in exp if got.get(n)!=exp[n]]
    print(f'{name}: {len(exp)-len(bad)}/{len(exp)} match' + ('' if not bad else f'  MISMATCH {bad[:3]}'))
    CORRECT = CORRECT and not bad
print('\nCORRECTNESS:', 'PASS' if CORRECT else 'FAIL - do NOT report performance')


## 2) Batch sweep - push past 1536 until allocation fails or throughput plateaus
Each nonce needs an 8 MiB scratchpad in global memory, so VRAM caps concurrency. Failures are reported honestly.


In [ ]:
import subprocess, re
assert CORRECT, 'correctness failed; refusing to benchmark'
batch_rows = []
for b in [1024, 1280, 1536, 1600, 1664, 1728, 1792, 1856, 1920, 2048]:
    o = subprocess.run(['./v2_gpu','sweep',str(b),'1','32'],capture_output=True,text=True).stdout
    m = re.search(r'hps=([0-9.]+) vram_gib=([0-9.]+)', o)
    if m:
        batch_rows.append((b, float(m.group(1)), float(m.group(2))))
        print(f'batch {b:5d}: {float(m.group(1)):9.2f} H/s   VRAM {float(m.group(2)):.2f} GiB')
    else:
        print(f'batch {b:5d}: {o.strip() or "no result"}')
best_batch = max(batch_rows, key=lambda r: r[1])[0] if batch_rows else 1536
print('\nbest batch by throughput:', best_batch)


## 3) Threads-per-block sweep at the best batch


In [ ]:
tpb_rows = []
for t in [32, 64, 128, 256]:
    o = subprocess.run(['./v2_gpu','sweep',str(best_batch),'1',str(t)],capture_output=True,text=True).stdout
    m = re.search(r'hps=([0-9.]+)', o)
    if m:
        tpb_rows.append((t, float(m.group(1))))
        print(f'tpb {t:4d}: {float(m.group(1)):9.2f} H/s')
    else:
        print(f'tpb {t:4d}: {o.strip() or "no result"}')
best_tpb = max(tpb_rows, key=lambda r: r[1])[0] if tpb_rows else 32
# also confirm interleaved still beats contiguous at the chosen config
for mp,nm in [(0,'contiguous'),(1,'interleaved')]:
    o = subprocess.run(['./v2_gpu','sweep',str(best_batch),str(mp),str(best_tpb)],capture_output=True,text=True).stdout
    m = re.search(r'hps=([0-9.]+)', o)
    if m: print(f'{nm}: {float(m.group(1)):9.2f} H/s')
print('\nbest tpb:', best_tpb)


## 4) Long steady-state trials (>=5 x >=30 s) at the best configuration


In [ ]:
import statistics
trials = []
for i in range(5):
    o = subprocess.run(['./v2_gpu','steady',str(best_batch),'1',str(best_tpb),'30'],capture_output=True,text=True).stdout
    m = re.search(r'hps=([0-9.]+)', o)
    if m:
        trials.append(float(m.group(1))); print(f'trial {i+1}: {trials[-1]:.2f} H/s')
trials.sort()
if trials:
    print(f'\nGPU steady H/s  p50={statistics.median(trials):.2f}  min={trials[0]:.2f}  max={trials[-1]:.2f}  n={len(trials)}')


## 5) Active GPU power (energy-counter delta preferred; sampling gated to the hashing window)


In [ ]:
# Active-power measurement, gated to the hashing window only.
# Prefers the GPU total-energy counter (nvidia-smi total_energy_consumption, mJ) and takes a delta
# across the steady window; falls back to dense power.draw sampling with startup/shutdown discarded.
import subprocess, threading, time, re, statistics, json

def smi(q):
    try:
        return subprocess.run(['nvidia-smi', f'--query-gpu={q}', '--format=csv,noheader,nounits'],
                              capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        return ''

HAS_ENERGY = False
try:
    v = smi('total_energy_consumption')
    HAS_ENERGY = bool(v) and 'Supported' not in v and float(v.split('\n')[0]) >= 0
except Exception:
    HAS_ENERGY = False
print('energy counter available:', HAS_ENERGY)

def run_steady_with_power(batch, mapping, tpb, secs):
    samples, active = [], {'on': False}
    e0 = {'v': None}; e1 = {'v': None}
    def poll():
        while not active['on']:
            time.sleep(0.02)
        if HAS_ENERGY:
            try: e0['v'] = float(smi('total_energy_consumption').split('\n')[0])
            except Exception: pass
        while active['on']:
            row = smi('power.draw,utilization.gpu,clocks.sm,clocks.mem,temperature.gpu')
            if row:
                try:
                    p,u,cs,cm,t = [float(x) for x in row.split('\n')[0].split(',')]
                    samples.append((p,u,cs,cm,t))
                except Exception: pass
            time.sleep(0.1)
        if HAS_ENERGY:
            try: e1['v'] = float(smi('total_energy_consumption').split('\n')[0])
            except Exception: pass

    th = threading.Thread(target=poll); th.start()
    proc = subprocess.Popen(['./v2_gpu','steady',str(batch),str(mapping),str(tpb),str(secs)],
                            stdout=subprocess.PIPE, text=True, bufsize=1)
    out_lines = []
    for line in proc.stdout:
        out_lines.append(line)
        if 'STEADY_BEGIN' in line: active['on'] = True      # power window OPENS (post-warmup)
        if 'STEADY_END'   in line: active['on'] = False     # power window CLOSES
    proc.wait(); active['on'] = False; th.join()
    out = ''.join(out_lines)
    m = re.search(r'STEADY .*hps=([0-9.]+) vram_gib=([0-9.]+)', out)
    hps = float(m.group(1)) if m else 0.0
    vram = float(m.group(2)) if m else 0.0

    # discard first/last 10% of samples (ramp) -- the window is already gated, this is belt-and-braces
    s = samples[len(samples)//10 : len(samples) - len(samples)//10] or samples
    res = {'hps': hps, 'vram_gib': vram, 'n_samples': len(s)}
    if s:
        pw = [x[0] for x in s]
        res.update(avg_w=statistics.mean(pw), med_w=statistics.median(pw), peak_w=max(pw),
                   util=statistics.mean([x[1] for x in s]), sm_mhz=statistics.mean([x[2] for x in s]),
                   mem_mhz=statistics.mean([x[3] for x in s]), temp_c=statistics.mean([x[4] for x in s]))
    if HAS_ENERGY and e0['v'] is not None and e1['v'] is not None and secs > 0:
        res['energy_delta_j'] = (e1['v'] - e0['v']) / 1000.0
        res['energy_avg_w'] = res['energy_delta_j'] / secs
    return res


In [ ]:
res = run_steady_with_power(best_batch, 1, best_tpb, 45)
import json; print(json.dumps(res, indent=1))
W = res.get('energy_avg_w') or res.get('med_w')
if W: print(f"\nACTIVE power {W:.1f} W  ->  GPU H/s per watt = {res['hps']/W:.3f}")
print('source:', 'energy counter delta' if 'energy_avg_w' in res else 'sampled power.draw (median, gated+trimmed)')


## 6) Occupancy and limiters


In [ ]:
import os
if os.path.exists('./v2_occ'):
    for t in sorted({32, best_tpb}):
        print(subprocess.run(['./v2_occ',str(t)],capture_output=True,text=True).stdout)
else:
    print('occupancy reporter unavailable (its build failed); ptxas stats below')
!nvcc -O3 -arch=sm_75 --ptxas-options=-v -c v2_gpu.cu -o /dev/null 2>&1 | grep -E 'Function|registers|stack|spill|gmem|lmem' | head -20


## 7) Colab CPU reference (SECONDARY only - does not replace the local CPU comparison)


In [ ]:
%%bash
cat > cpu_ref.cpp <<'EOF'
#define MEEP_HOST_ONLY
#include "v2_cuda_core.cuh"
#include <cstdio>
#include <vector>
#include <chrono>
int main(){ MeepKeys K; meep_init_keys(&K); uint8_t ek[32],sh[32];
  for(int i=0;i<32;++i){ek[i]=(uint8_t)(i*7+1);sh[i]=(uint8_t)(i*3+9);} uint8_t t[8]={1,2,3,4,5,6,7,8};
  std::vector<uint64_t> D(DATASET_WORDS); meep_v2_build_dataset(&K,ek,D.data());
  std::vector<MInstr> p(PROG_LEN); { std::vector<uint8_t> pb(PROG_LEN*8); uint8_t fb[256];
    meep_derive_program(&K,ek,sh,4096,p.data(),pb.data(),fb); }
  std::vector<uint64_t> SP(SCRATCH_WORDS); std::vector<uint8_t> fb(1024); uint8_t o[32];
  meep_v2_hash_one(&K,D.data(),p.data(),t,8,0,SP.data(),1,fb.data(),o);
  auto a=std::chrono::steady_clock::now(); int N=30;
  for(int i=0;i<N;++i) meep_v2_hash_one(&K,D.data(),p.data(),t,8,i+1,SP.data(),1,fb.data(),o);
  double s=std::chrono::duration<double>(std::chrono::steady_clock::now()-a).count();
  printf("colab CPU 1-thread: %.2f H/s\n", N/s); return 0; }
EOF
g++ -O3 -std=c++17 cpu_ref.cpp -o cpu_ref && ./cpu_ref


## 8) Final GPU report - paste these back

```
GPU model / CUDA / driver      : ___
Correctness (both mappings)    : PASS/FAIL
Best batch / best tpb          : ___ / ___
VRAM used                      : ___ GiB
Steady H/s p50 (min-max, n=5)  : ___
ACTIVE power (source)          : ___ W (energy-counter | sampled)
avg / median / peak W          : ___
GPU util / SM MHz / mem MHz / T : ___
GPU H/s per watt               : ___
Occupancy (achieved/theoretical): ___
Registers / local mem per thread: ___
Main limiter                   : scratchpad capacity | registers | latency | bandwidth
Colab CPU 1-thread (secondary) : ___
```

**Efficiency gate.** GPU H/s-per-watt divided by CPU H/s-per-watt must be **<= 3x** (pre-registered, not to be changed after seeing results). The CPU denominator is measured locally on the Ryzen AI 7 PRO 350; if CPU power cannot be measured reliably the gate is **INCONCLUSIVE**, never PASS.
